In [1]:
"""
PRODUCTION SCRIPT: Parallel PostgreSQL to Spark
Reads all tables from Data_engineering_practice database
Handles errors, retries, and provides comprehensive logging
"""

import os
import sys
import time
import logging
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql import SparkSession
from pyspark.sql.utils import AnalysisException

# ============================================================================
# CONFIGURATION - UPDATE THESE VALUES
# ============================================================================

# Database configuration
DB_CONFIG = {
    "url": "jdbc:postgresql://localhost:5432/Data_engineering_practice",
    "user": "postgres",
    "password": "xxxxxxxx",  # 🔴 CHANGE THIS
    "driver": "org.postgresql.Driver",
    "fetchsize": "5000"  # Optimize for large tables
}

# Paths
BASE_DIR = "/home/odinsbeard/Data_engineering_Journey/week6_spark"
JAR_PATH = f"{BASE_DIR}/jars/postgresql-42.7.1.jar"
OUTPUT_DIR = f"{BASE_DIR}/data/postgres_export"
LOG_DIR = f"{BASE_DIR}/logs"

# Tables to read
TABLES = [
    "customers",
    "products",
    "orders",
    "order_items",
    "sales_reps",
    "customer_rep_assignment"
]

# Performance settings
MAX_WORKERS = 4  # Parallel connections
RETRY_COUNT = 2  # Number of retries on failure

# ============================================================================
# SETUP LOGGING
# ============================================================================

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

log_filename = f"{LOG_DIR}/postgres_export_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_filename),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)
logger

<Logger __main__ (INFO)>

In [2]:
# ============================================================================
# STEP 1: INITIALIZE SPARK WITH POSTGRESQL DRIVER
# ============================================================================

def init_spark():
    """Initialize Spark session with PostgreSQL driver"""
    logger.info("=" * 70)
    logger.info("INITIALIZING SPARK SESSION")
    logger.info("=" * 70)
    
    # Verify JAR exists
    if not os.path.exists(JAR_PATH):
        logger.error(f"PostgreSQL driver not found at: {JAR_PATH}")
        logger.error("Please download it first:")
        logger.error(f"  mkdir -p {BASE_DIR}/jars")
        logger.error(f"  cd {BASE_DIR}/jars")
        logger.error("  wget https://jdbc.postgresql.org/download/postgresql-42.7.1.jar")
        sys.exit(1)
    
    logger.info(f"✅ Driver found: {JAR_PATH}")
    
    # Create Spark session
    try:
        spark = SparkSession.builder \
            .appName("PostgreSQL to Spark Parallel Load") \
            .config("spark.jars", JAR_PATH) \
            .config("spark.driver.extraClassPath", JAR_PATH) \
            .config("spark.sql.adaptive.enabled", "true") \
            .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
            .config("spark.sql.adaptive.skewJoin.enabled", "true") \
            .config("spark.sql.files.maxPartitionBytes", "128MB") \
            .getOrCreate()
        
        spark.sparkContext.setLogLevel("WARN")
        logger.info(f"✅ Spark session created (version: {spark.version})")
        return spark
        
    except Exception as e:
        logger.error(f"❌ Failed to create Spark session: {e}")
        sys.exit(1)

# ============================================================================
# STEP 2: FUNCTION TO READ A SINGLE TABLE WITH RETRY LOGIC
# ============================================================================

def read_table(spark, table_name, retry_count=RETRY_COUNT):
    """
    Read a table from PostgreSQL with retry logic
    """
    start_time = time.time()
    logger.info(f"⏳ Reading table: {table_name}")
    
    for attempt in range(retry_count + 1):
        try:
            # Read table
            df = spark.read.jdbc(
                url=DB_CONFIG["url"],
                table=table_name,
                properties={
                    "user": DB_CONFIG["user"],
                    "password": DB_CONFIG["password"],
                    "driver": DB_CONFIG["driver"],
                    "fetchsize": DB_CONFIG["fetchsize"]
                }
            )
            
            # Get row count
            row_count = df.count()
            
            # Save as Parquet (partitioned for performance)
            output_path = f"{OUTPUT_DIR}/{table_name}"
            df.write.mode("overwrite") \
                .option("compression", "snappy") \
                .parquet(output_path)
            
            # Register temp view
            df.createOrReplaceTempView(table_name)
            
            elapsed = time.time() - start_time
            logger.info(f"✅ {table_name}: {row_count:,} rows loaded in {elapsed:.2f}s")
            return {
                "table": table_name,
                "status": "success",
                "rows": row_count,
                "time": elapsed,
                "output": output_path
            }
            
        except Exception as e:
            if attempt < retry_count:
                logger.warning(f"⚠️ Attempt {attempt + 1} failed for {table_name}, retrying... ({e})")
                time.sleep(2 ** attempt)  # Exponential backoff
            else:
                elapsed = time.time() - start_time
                logger.error(f"❌ {table_name}: FAILED after {elapsed:.2f}s - {str(e)}")
                return {
                    "table": table_name,
                    "status": "failed",
                    "error": str(e),
                    "time": elapsed
                }


# ============================================================================
# STEP 3: READ TABLES IN PARALLEL
# ============================================================================

def load_all_tables(spark):
    """Load all tables in parallel"""
    logger.info("=" * 70)
    logger.info("PARALLEL TABLE LOADING")
    logger.info("=" * 70)
    logger.info(f"🔄 Using {MAX_WORKERS} parallel connections")
    logger.info(f"📋 Tables to load: {', '.join(TABLES)}")
    
    start_total = time.time()
    results = []
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Submit all tasks
        future_to_table = {
            executor.submit(read_table, spark, table): table 
            for table in TABLES
        }
        
        # Collect results as they complete
        for future in as_completed(future_to_table):
            table = future_to_table[future]
            try:
                result = future.result()
                results.append(result)
            except Exception as e:
                logger.error(f"❌ Unexpected error on {table}: {e}")
                results.append({
                    "table": table,
                    "status": "failed",
                    "error": str(e)
                })
    
    total_time = time.time() - start_total
    
    # Calculate statistics
    successful = [r for r in results if r["status"] == "success"]
    failed = [r for r in results if r["status"] == "failed"]
    total_rows = sum(r.get("rows", 0) for r in successful)
    
    logger.info("=" * 70)
    logger.info("LOADING COMPLETE - SUMMARY")
    logger.info("=" * 70)
    logger.info(f"✅ Successful: {len(successful)}/{len(TABLES)} tables")
    logger.info(f"❌ Failed: {len(failed)}/{len(TABLES)} tables")
    logger.info(f"📊 Total rows loaded: {total_rows:,}")
    logger.info(f"⏱️  Total time: {total_time:.2f} seconds")
    
    if successful:
        logger.info("\n📈 Successful tables:")
        for r in successful:
            logger.info(f"  ✅ {r['table']}: {r['rows']:,} rows in {r['time']:.2f}s")
    
    if failed:
        logger.info("\n❌ Failed tables:")
        for r in failed:
            logger.info(f"  ❌ {r['table']}: {r.get('error', 'Unknown error')}")
    
    return results, successful, failed


# ============================================================================
# STEP 4: VERIFY LOADED DATA
# ============================================================================

def verify_data(spark, successful_tables):
    """Verify that loaded tables are accessible in Spark"""
    logger.info("=" * 70)
    logger.info("VERIFYING LOADED DATA")
    logger.info("=" * 70)
    
    verified = []
    failed_verification = []
    
    for table_info in successful_tables:
        table = table_info["table"]
        try:
            count = spark.sql(f"SELECT COUNT(*) FROM {table}").collect()[0][0]
            expected = table_info["rows"]
            if count == expected:
                logger.info(f"✅ {table}: {count:,} rows (verified)")
                verified.append(table)
            else:
                logger.warning(f"⚠️ {table}: Count mismatch - Expected {expected:,}, Got {count:,}")
                failed_verification.append(table)
        except Exception as e:
            logger.error(f"❌ {table}: Verification failed - {e}")
            failed_verification.append(table)
    
    return verified, failed_verification


# ============================================================================
# STEP 5: CREATE ENRICHED VIEWS (FIXED VERSION)
# ============================================================================

def create_enriched_views(spark, successful_tables):
    """Create joined views if all required tables are available"""
    required_for_analysis = {"customers", "orders", "order_items", "products"}
    
    if required_for_analysis.issubset(set(successful_tables)):
        logger.info("=" * 70)
        logger.info("CREATING ENRICHED VIEWS")
        logger.info("=" * 70)
        
        try:
            # First, let's check the schemas to verify column names
            logger.info("📋 Verifying column names...")
            
            # Show order_items schema for debugging
            order_items_schema = spark.table("order_items").columns
            logger.info(f"order_items columns: {order_items_schema}")
            
            products_schema = spark.table("products").columns
            logger.info(f"products columns: {products_schema}")
            
            # Dynamically build the query based on actual column names
            # This makes the script more robust to schema changes
            
            # Determine the price column name in order_items
            if "unit_price" in order_items_schema:
                order_price_col = "unit_price"
            elif "price" in order_items_schema:
                order_price_col = "price"
            else:
                order_price_col = "unit_price"  # default
            
            # Determine the price column name in products
            if "price" in products_schema:
                product_price_col = "price"
            elif "unit_price" in products_schema:
                product_price_col = "unit_price"
            else:
                product_price_col = "price"  # default
            
            logger.info(f"Using '{order_price_col}' from order_items and '{product_price_col}' from products")
            
            # Create enriched customer orders view with correct column names
            spark.sql(f"""
                CREATE OR REPLACE TEMP VIEW enriched_orders AS
                SELECT 
                    o.order_id,
                    o.order_date,
                    o.status,
                    c.customer_id,
                    c.first_name,
                    c.last_name,
                    c.email,
                    c.country,
                    p.product_id,
                    p.product_name,
                    p.category,
                    p.{product_price_col} as unit_price,
                    oi.quantity,
                    oi.{order_price_col} as sale_price,
                    oi.quantity * oi.{order_price_col} as line_total
                FROM orders o
                JOIN customers c ON o.customer_id = c.customer_id
                JOIN order_items oi ON o.order_id = oi.order_id
                JOIN products p ON oi.product_id = p.product_id
            """)
            
            # Save enriched data
            enriched_df = spark.table("enriched_orders")
            enriched_path = f"{OUTPUT_DIR}/enriched_orders"
            enriched_df.write.mode("overwrite") \
                .option("compression", "snappy") \
                .parquet(enriched_path)
            
            row_count = enriched_df.count()
            logger.info(f"✅ Enriched view created with {row_count:,} rows")
            logger.info(f"💾 Saved to: {enriched_path}")
            
            # Show sample
            logger.info("\n📊 Sample enriched data:")
            enriched_df.show(5, truncate=False)
            
        except Exception as e:
            logger.error(f"❌ Failed to create enriched views: {e}")
            # Print the full stack trace for debugging
            import traceback
            logger.error(traceback.format_exc())
    else:
        logger.warning("⚠️ Insufficient tables for enriched views")
        logger.warning(f"   Required: {required_for_analysis}")
        logger.warning(f"   Available: {set(successful_tables)}")

        

In [3]:
# ============================================================================
# STEP 6: MAIN EXECUTION
# ============================================================================

def main():
    """Main execution function"""
    start_program = time.time()
    
    try:
        # Initialize Spark
        spark = init_spark()
        
        # Load all tables
        results, successful, failed = load_all_tables(spark)
        
        # Verify loaded data
        if successful:
            verified, failed_verification = verify_data(spark, successful)
        else:
            verified = []
        
        # Create enriched views if possible
        if verified:
            successful_tables = [r["table"] for r in successful if r["table"] in verified]
            create_enriched_views(spark, successful_tables)
        
        # Final summary
        total_time = time.time() - start_program
        logger.info("=" * 70)
        logger.info("PROGRAM COMPLETE")
        logger.info("=" * 70)
        logger.info(f"⏱️  Total program time: {total_time:.2f} seconds")
        logger.info(f"📁 Output directory: {OUTPUT_DIR}")
        logger.info(f"📝 Log file: {log_filename}")
        
        # Return exit code
        if failed:
            logger.warning(f"⚠️ {len(failed)} tables failed to load")
            sys.exit(1)
        else:
            logger.info("✅ ALL TABLES LOADED SUCCESSFULLY!")
            sys.exit(0)
            
    except Exception as e:
        logger.error(f"❌ Fatal error: {e}")
        sys.exit(1)
    finally:
        # Always stop Spark session
        if 'spark' in locals():
            spark.stop()
            logger.info("✅ Spark session stopped")

# ============================================================================
# ENTRY POINT
# ============================================================================

if __name__ == "__main__":
    main()



2026-03-13 23:42:01,368 - INFO - ======================================================================
2026-03-13 23:42:01,376 - INFO - INITIALIZING SPARK SESSION
2026-03-13 23:42:01,381 - INFO - ======================================================================
2026-03-13 23:42:01,381 - INFO - ✅ Driver found: /home/odinsbeard/Data_engineering_Journey/week6_spark/jars/postgresql-42.7.1.jar


26/03/13 23:42:07 WARN Utils: Your hostname, odinsbeard-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/03/13 23:42:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/03/13 23:42:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2026-03-13 23:42:13,448 - INFO - ✅ Spark session created (version: 3.5.0)
2026-03-13 23:42:13,450 - INFO - ======================================================================
2026-03-13 23:42:13,454 - INFO - PARALLEL TABLE LOADING
2026-03-13 23:42:13,456 - INFO - ======================================================================
2026-03-13 23:42:13,457 - INFO - 🔄 Using 4 parallel connections
2026-03-13 23:42:13,465 - INFO - 📋 Tables to load: customers, products, orders, order_items, sales_reps, customer_rep_assignment
2026-03-13 23:42:13,466 - INFO - ⏳ Reading table: customers
2026-03-13 23:42:13,467 - INFO - ⏳ Reading table: products
2026-03-13 23:42:13,472 - INFO - ⏳ Reading table: orders
2026-03-13 23:42:13,473 - INFO - ⏳ Reading table: order_items


2026-03-13 23:42:28,994 - INFO - ✅ customers: 101 rows loaded in 15.53s
2026-03-13 23:42:28,999 - INFO - ⏳ Reading table: sales_reps
2026-03-13 23:42:28,997 - INFO - ✅ order_items: 721 rows loaded in 15.52s
2026-03-13 23:42:29,006 - INFO - ⏳ Reading table: customer_rep_assignment
2026-03-13 23:42:28,994 - INFO - ✅ orders: 200 rows loaded in 15.52s
2026-03-13 23:42:28,997 - INFO - ✅ products: 50 rows loaded in 15.53s
2026-03-13 23:42:30,642 - INFO - ✅ customer_rep_assignment: 100 rows loaded in 1.64s
2026-03-13 23:42:30,791 - INFO - ✅ sales_reps: 10 rows loaded in 1.79s
2026-03-13 23:42:30,807 - INFO - Closing down clientserver connection
2026-03-13 23:42:30,829 - INFO - Closing down clientserver connection
2026-03-13 23:42:30,846 - INFO - Closing down clientserver connection
2026-03-13 23:42:30,840 - INFO - Closing down clientserver connection
2026-03-13 23:42:30,873 - INFO - ======================================================================
2026-03-13 23:42:30,875 - INFO - LOADING

2026-03-13 23:42:38,001 - INFO - ✅ Enriched view created with 721 rows
2026-03-13 23:42:38,003 - INFO - 💾 Saved to: /home/odinsbeard/Data_engineering_Journey/week6_spark/data/postgres_export/enriched_orders
2026-03-13 23:42:38,011 - INFO - 
📊 Sample enriched data:
+--------+--------------------------+---------+-----------+-----------+----------+--------------------+-------+----------+------------+-------------+----------+--------+----------+----------+
|order_id|order_date                |status   |customer_id|first_name |last_name |email               |country|product_id|product_name|category     |unit_price|quantity|sale_price|line_total|
+--------+--------------------------+---------+-----------+-----------+----------+--------------------+-------+----------+------------+-------------+----------+--------+----------+----------+
|148     |2025-02-07 07:08:02.767167|Completed|83         |FirstName83|LastName83|customer83@email.com|USA    |31        |Product 31  |Clothing     |143.74    

SystemExit: 0

/home/odinsbeard/Data_engineering_Journey/week6_spark/venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
